# Frontend Pipeline For Qwen3 MoE

实现 notebook 里定义的 Qwen3 MoE 前端全流程，并最终生成 `macro_op_list`。


## 流程
- 初始化 `NandConfig`、`InferenceConfig`、`MoEParallelConfig`
- 加载 `qwen3-moe-235B` model card
- 初始化 `Qwen3MoEDecoderLayer`
- 使用 `NxTracer` trace 得到计算图
- 运行 `NormalizePass` 获得简化图
- 用 `build_kv_cache_state` 构造 `KVCacheState`
- 构造 `NxGraphMeta` 并写入 `graph.meta`
- 运行 `CodeGenPass` 生成 `macro_op_list`
- 打印完整指令列表，不进行 cycle 仿真


In [22]:
import json
from collections import Counter
from pathlib import Path

import torch
from torch.fx import GraphModule

from nandmachine.commands.macro import All2AllOp, MatMulOp, VectorOp
from nandmachine.config.config import NandConfig
from nandmachine.config.inference_config import InferenceConfig, MoEParallelConfig
from nandmachine.frontend.core.graph.base import NxGraphMeta, NxTracer
from nandmachine.frontend.core.passes.cod_gen import CodeGenPass
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.qwen3_moe import Qwen3MoEDecoderLayer
from nandmachine.frontend.utlis import build_kv_cache_state

MODEL_CARD_PATH = Path("model_cards/qwen3-moe-235B.json")
assert MODEL_CARD_PATH.exists(), MODEL_CARD_PATH


def node_summary(graph_module: GraphModule) -> list[tuple[str, str]]:
    return [(node.op, str(node.target)) for node in graph_module.graph.nodes]


def macro_summary(macro_op_list: list[object]) -> Counter:
    return Counter(type(op).__name__ for op in macro_op_list)


In [23]:
model_card = json.loads(MODEL_CARD_PATH.read_text())
model_card.setdefault("attention_type", "gqa")
model_config = type("Qwen3MoeConfig", (), model_card)()
nand_config = NandConfig(
    num_channels=1,
    num_plane=1,
    num_block=4,
    num_pages=16,
    tRead=1.0,
    tWrite=2.0,
    tErase=3.0,
    page_size=4,
    sram_threshold=1024*1024*20,
)
parallel_config = MoEParallelConfig(
    attn_dp_size=8,
    attn_tp_size=1,
    ffn_tp_size=4,
    ffn_ep_size=2,
)
inference_config = InferenceConfig(
    batch_size=2,
    input_sequence_length=8,
    output_sequence_length=4,
    weight_bits=16,
    activation_bits=16,
    kv_cache_bits=16,
    kv_block_size_bytes=1024,
    parallel_config=parallel_config,
)
kv_cache_state = build_kv_cache_state(nand_config, model_config, inference_config)
graph_meta = NxGraphMeta(
    nand_config=nand_config,
    model_config=model_config,
    inference_config=inference_config,
    kv_cache_state=kv_cache_state,
)

print(model_config)
print(parallel_config)
print(kv_cache_state)


MoEParallelConfig(attn_dp_size=8, attn_tp_size=1, ffn_tp_size=4, ffn_ep_size=2)
KVCacheState(total_kv_cache_size_per_layer=24576, num_nand_pages_per_layer=6, num_hyper_pages_per_layer=6, kv_block_size_tokens=1, num_kv_blocks=24, kv_cache_num_pages_per_layer=6)


In [24]:
with torch.device("meta"):
    model = Qwen3MoEDecoderLayer(model_config, parallel_config)
    graph = NxTracer().trace(model)
    graph_module = GraphModule(model, graph)

NormalizePass().transform(graph_module)
graph_module.graph.meta = {CodeGenPass.GRAPH_META_KEY: graph_meta}
CodeGenPass().transform(graph_module)

macro_op_list = graph_module.graph.meta[CodeGenPass.MACRO_OP_LIST_META_KEY]
vector_ops = [op for op in macro_op_list if isinstance(op, VectorOp)]
all_to_all_ops = [op for op in macro_op_list if isinstance(op, All2AllOp)]

print("normalized nodes:", node_summary(graph_module))
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("all_to_all count:", len(all_to_all_ops))
print("vector op summary:", [(op.vector_op_type, op.vector_shape) for op in vector_ops])


normalized nodes: [('placeholder', 'hidden_states'), ('call_module', 'input_layernorm'), ('call_module', 'self_attn.qkv_proj'), ('call_module', 'self_attn.q_norm'), ('call_module', 'self_attn.k_norm'), ('get_attr', 'self_attn._dummy_positions'), ('call_module', 'self_attn.rotary_emb'), ('call_module', 'self_attn.attn'), ('call_module', 'self_attn.o_proj'), ('call_function', '<built-in function add>'), ('call_module', 'post_attention_layernorm'), ('call_module', 'mlp'), ('call_function', '<built-in function add>')]
macro op count: 532
macro op type summary: {'VectorOp': 70, 'SramPrefetch': 132, 'MatMulOp': 131, 'SramPrefetchRelease': 132, 'FlashAttnOp': 1, 'All2AllOp': 2, 'AllReduceOp': 64}
all_to_all count: 2
vector op summary: [('rms_norm', [2, 4096]), ('rms_norm', [2, 128]), ('rms_norm', [2, 128]), ('rms_norm', [2, 4096]), ('moe_topk_router', [2, 128, 8]), ('silu_mul', [1, 384]), ('silu_mul', [1, 384]), ('silu_mul', [1, 384]), ('silu_mul', [1, 384]), ('silu_mul', [1, 384]), ('silu_mu

In [25]:
assert macro_op_list
assert any(isinstance(op, MatMulOp) for op in macro_op_list)
assert any(isinstance(op, All2AllOp) for op in macro_op_list)
assert any(op.vector_op_type == "moe_topk_router" for op in vector_ops)
assert any(op.vector_op_type == "moe_weighted_sum" for op in vector_ops)
assert any(op.vector_op_type == "silu_mul" for op in vector_ops)

print("frontend moe pipeline completed successfully")


frontend moe pipeline completed successfully


In [26]:
print(macro_op_list)


[VectorOp(id=3535, input_ops=[], vector_op_type='rms_norm', vector_shape=[2, 4096]), SramPrefetch(id=3536, input_ops=[], num_prefetch_pages=18432), MatMulOp(id=3537, input_ops=[SramPrefetch(id=3536, input_ops=[], num_prefetch_pages=18432)], dim=(2, 4096, 9216), weight_bits=16), SramPrefetchRelease(id=3538, input_ops=[MatMulOp(id=3537, input_ops=[SramPrefetch(id=3536, input_ops=[], num_prefetch_pages=18432)], dim=(2, 4096, 9216), weight_bits=16)]), VectorOp(id=3539, input_ops=[], vector_op_type='rms_norm', vector_shape=[2, 128]), VectorOp(id=3540, input_ops=[], vector_op_type='rms_norm', vector_shape=[2, 128]), SramPrefetch(id=3541, input_ops=[], num_prefetch_pages=1), FlashAttnOp(id=3542, input_ops=[SramPrefetch(id=3541, input_ops=[], num_prefetch_pages=1)], qk_bmm_shape=(16, 16, 128, 1), sv_bmm_shape=(16, 16, 1, 128), softmax_shape=(256, 1), weight_bits=16), SramPrefetchRelease(id=3543, input_ops=[FlashAttnOp(id=3542, input_ops=[SramPrefetch(id=3541, input_ops=[], num_prefetch_pages=1

In [27]:
for inst in macro_op_list:
    print(inst)


VectorOp(id=3535, input_ops=[], vector_op_type='rms_norm', vector_shape=[2, 4096])
SramPrefetch(id=3536, input_ops=[], num_prefetch_pages=18432)
MatMulOp(id=3537, input_ops=[SramPrefetch(id=3536, input_ops=[], num_prefetch_pages=18432)], dim=(2, 4096, 9216), weight_bits=16)
SramPrefetchRelease(id=3538, input_ops=[MatMulOp(id=3537, input_ops=[SramPrefetch(id=3536, input_ops=[], num_prefetch_pages=18432)], dim=(2, 4096, 9216), weight_bits=16)])
VectorOp(id=3539, input_ops=[], vector_op_type='rms_norm', vector_shape=[2, 128])
VectorOp(id=3540, input_ops=[], vector_op_type='rms_norm', vector_shape=[2, 128])
SramPrefetch(id=3541, input_ops=[], num_prefetch_pages=1)
FlashAttnOp(id=3542, input_ops=[SramPrefetch(id=3541, input_ops=[], num_prefetch_pages=1)], qk_bmm_shape=(16, 16, 128, 1), sv_bmm_shape=(16, 16, 1, 128), softmax_shape=(256, 1), weight_bits=16)
SramPrefetchRelease(id=3543, input_ops=[FlashAttnOp(id=3542, input_ops=[SramPrefetch(id=3541, input_ops=[], num_prefetch_pages=1)], qk_bm

In [28]:
len(macro_op_list)


532